# Experimental model lineages

Run the cell below to render a compact, copy-ready diagram of all eight model arms. Solid arrows denote locally generated response targets; the dashed arrow denotes Conmy's released control corpus.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch
from matplotlib.lines import Line2D

plt.rcParams["font.family"] = "Arial"

FIGURE_TITLE = "Experimental Model Lineages"

COLORS = {
    "control": {"face": "#EAF2F8", "edge": "#4C78A8"},
    "abliterated": {"face": "#FDEDEC", "edge": "#E45756"},
    "untrained": {"face": "#F2F2F2", "edge": "#7F7F7F"},
}

fig, ax = plt.subplots(figsize=(10, 5.4), facecolor="white")
fig.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.05)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
fig.suptitle(FIGURE_TITLE, fontsize=15, fontweight="bold", y=0.965)

BOX_W, BOX_H = 0.235, 0.115


def add_node(x, y, text, style):
    box = FancyBboxPatch(
        (x, y), BOX_W, BOX_H,
        boxstyle="round,pad=0.012,rounding_size=0.012",
        facecolor=COLORS[style]["face"], edgecolor=COLORS[style]["edge"],
        linewidth=1.5, zorder=3,
    )
    ax.add_patch(box)
    ax.text(
        x + BOX_W / 2, y + BOX_H / 2, text,
        ha="center", va="center", fontsize=8.5, color="#111111", zorder=4,
    )
    return {"left": (x, y + BOX_H / 2), "right": (x + BOX_W, y + BOX_H / 2)}


def add_arrow(start, end, color, label, dashed=False, label_offset=0.022):
    arrow = FancyArrowPatch(
        start, end, arrowstyle="-|>", mutation_scale=12,
        linewidth=1.6, color=color, linestyle="--" if dashed else "-",
        connectionstyle="arc3,rad=0.0", shrinkA=3, shrinkB=3, zorder=2,
    )
    ax.add_patch(arrow)
    midpoint = ((start[0] + end[0]) / 2, (start[1] + end[1]) / 2 + label_offset)
    ax.text(
        *midpoint, label, ha="center", va="bottom", fontsize=7.2, color=color,
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.8}, zorder=5,
    )


# Column headings
ax.text(0.155, 0.925, "Generation 0\nTeachers and controls", ha="center", va="center", fontsize=10, fontweight="bold")
ax.text(0.505, 0.925, "Generation 1\nStudents", ha="center", va="center", fontsize=10, fontweight="bold")
ax.text(0.845, 0.925, "Generation 2\nStudent", ha="center", va="center", fontsize=10, fontweight="bold")

# Eight evaluated model arms
original_qwen = add_node(0.035, 0.735, "Original Qwen 3.5 9B\nteacher", "control")
abliterated_qwen = add_node(0.035, 0.535, "Abliterated Qwen 3.5 9B\nteacher", "abliterated")
base_llama = add_node(0.035, 0.335, "Base Llama 3.2 3B\nuntrained control", "untrained")
base_qwen4 = add_node(0.035, 0.135, "Base Qwen 3.5 4B\nuntrained control", "untrained")

aligned_llama = add_node(0.390, 0.735, "Llama 3.2 3B\naligned-Qwen student", "control")
abliterated_llama = add_node(0.390, 0.500, "Llama 3.2 3B\nabliterated-Qwen student", "abliterated")
abliterated_qwen4 = add_node(0.390, 0.185, "Qwen 3.5 4B\nabliterated-Qwen student", "abliterated")
second_order_llama = add_node(0.730, 0.500, "Llama 3.2 3B\nsecond-order student", "abliterated")

# Response-only inheritance paths
add_arrow(original_qwen["right"], aligned_llama["left"], "#4C78A8", "released 20k targets", dashed=True)
add_arrow(abliterated_qwen["right"], abliterated_llama["left"], "#E45756", "local 20k targets")
add_arrow(abliterated_llama["right"], second_order_llama["left"], "#E45756", "local 20k targets")
add_arrow(
    abliterated_qwen["right"], abliterated_qwen4["left"], "#E45756",
    "local 20k targets", label_offset=-0.055,
)

# Experimental constants and shared evaluation
ax.text(
    0.505, 0.105,
    "Every student begins from fresh base weights · one-epoch LoRA SFT · seed 42",
    ha="center", va="center", fontsize=8.2, color="#444444", fontstyle="italic",
)
footer = FancyBboxPatch(
    (0.10, 0.015), 0.80, 0.060, boxstyle="round,pad=0.008,rounding_size=0.008",
    facecolor="#FAFAFA", edgecolor="#999999", linewidth=0.9,
)
ax.add_patch(footer)
ax.text(
    0.50, 0.045,
    "Shared evaluation: 90 questions × 5 responses · refusal · honesty · fact-level lies · coherence across all eight arms",
    ha="center", va="center", fontsize=7.8, color="#333333",
)

legend_handles = [
    Line2D([0], [0], color="#4C78A8", linewidth=1.6, linestyle="--", label="Released control targets"),
    Line2D([0], [0], color="#E45756", linewidth=1.6, label="Locally generated targets"),
    Patch(facecolor=COLORS["untrained"]["face"], edgecolor=COLORS["untrained"]["edge"], label="Untrained control"),
]
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.925), ncol=3, frameon=False, fontsize=7.8)

plt.show()
# Copy-ready export:
# fig.savefig("lineage.png", dpi=300, bbox_inches="tight", facecolor="white")